In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pymcts.games.bridgit.game import BridgitGame
from pymcts.games.bridgit.config import BoardConfig, NeuralNetConfig
from pymcts.games.bridgit.neural_net import BridgitNet
from pymcts.core.config import MCTSConfig, TrainingConfig
from pymcts.arena import SinglePlayerArena
from pymcts.arena.config import SinglePlayerArenaConfig
from pymcts.core.trainer import train

## Configure training

## Run training

In [ ]:
from pathlib import Path

board_config = BoardConfig(size=4)

net = BridgitNet(
        board_config=board_config, net_config=NeuralNetConfig(num_channels=64, num_res_blocks=4)
    )
net.load_checkpoint("trainings/run_2026-04-09_081744/iteration_009/post_training.pt")

game_factory = lambda: BridgitGame(config=board_config)
arena_config = SinglePlayerArenaConfig(num_games=100, swap_players=True)
self_play_arena = SinglePlayerArena(arena_config, game_factory, arena_dir=Path("trainings/self_play"))
eval_arena = SinglePlayerArena(arena_config, game_factory, arena_dir=Path("trainings/eval"))

train(
    game_factory=game_factory,
    net=BridgitNet(
        board_config=board_config, net_config=NeuralNetConfig(num_channels=64, num_res_blocks=4)
    ),
    mcts_config=MCTSConfig(num_simulations=2000, num_parallel_leaves=10),
    training_config=TrainingConfig(
        num_iterations=200, num_self_play_games=3000, self_play_batch_size=100
    ),
    self_play_arena=self_play_arena,
    eval_arena=eval_arena,
)